# Introduksjon til minste kvadraters kollokasjon og dennes anvendelse i fysisk geodesi - Spesialpensum H26

Datautforsking for minste kvadraters kollokasjon (LSC).

### Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

### File paths

In [2]:
# Path to data folder one level up from the notebooks folder
DATA_PATH = Path().cwd().parent / "data"
gravity_path = DATA_PATH / "gravity_disturbance_WHU-CASM-UGM2025_251_2159.xyz"  # gravity disturbance file
ha_path = DATA_PATH / "height_anomaly_WHU-CASM-UGM2025_251_2159.xyz"            # height anomaly file

### Reading the data

In [3]:
# Reading the gravity disturbance file (No header, four space-separated columns: lon, lat, h, value)
gravity_disturbance = pd.read_csv(gravity_path, sep=r"\s+", header=None, names=["lon", "lat", "h", "dg"])
height_anomaly = pd.read_csv(ha_path, sep=r"\s+", header=None, names=["lon", "lat", "h", "zeta"])

### Data exploration

In [4]:
# Summary statistics (mean of dg should be close to 0)
gravity_disturbance.describe()

,lon,lat,h,dg
count,27956.000000,27956.000000,27956.000000,27956.000000
mean,6.060000,62.203334,345.000074,0.279980
std,2.319045,1.116189,495.837324,19.723691
min,2.060000,60.286700,0.000000,-78.486320
25%,4.060000,61.244975,0.000000,-11.262482
50%,6.060000,62.203350,0.000000,-0.854425
75%,8.060000,63.161650,703.950000,10.310711
max,10.060000,64.120000,2137.800000,126.094393


In [5]:
# Plotting the distribution of the gravity disturbances
fig = px.histogram(
    gravity_disturbance,
    x="dg",
    nbins=100,
    title="Distribution of gravity disturbances",
    labels={"dg": "Gravity disturbance [mGal]"}
)
fig.update_layout(yaxis_title="Count")
fig.show()

In [6]:
# Reshaping dataframe to 2D grid (rows=lat, columns=lon, cells=dg)
grid = gravity_disturbance.pivot(index="lat", columns="lon", values="dg")

In [7]:
vmax = np.nanpercentile(np.abs(grid.values), 99)    # Color limit, ignores extreme values
mean_lat = grid.index.to_numpy().mean()             # Used for map proportions

fig = go.Figure(go.Heatmap(
    x=grid.columns,
    y=grid.index,
    z=grid.values,
    colorscale="RdBu_r",
    zmin=-vmax, zmax=vmax,  # Symmetric colorscale, white = 0
    colorbar=dict(title="Gravity disturbance [mGal]"),
    hovertemplate="Lon: %{x:.4f}°<br>Lat: %{y:.4f}°<br>δg: %{z:.3f} mGal<extra></extra>",
))

fig.update_layout(
    title="Gravity disturbance (degree 251-2159)",
    xaxis_title="Longitude [degrees]",
    yaxis_title="Latitude [degrees]",
    width=800, height=700,
)

# Correct east-west stretching at high latitudes
fig.update_yaxes(scaleanchor="x", scaleratio=1 / np.cos(np.deg2rad(mean_lat)))

fig.show()